### Load libraries

In [2]:
from typing import Iterator
import datasets
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

c:\Homework\Code File\Python Code File\Pratice\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load dataset

In [2]:
# Load FineWeb 10B sample (using only a slice for demo to save memory)
dataset = datasets.load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)

def get_texts(dataset: datasets.Dataset, limit: int = 100_000) -> Iterator[str]:
    # Get texts from the dataset until it reached the limit or is exhausted
    count = 0
    for sample in dataset:
        yield sample["text"]
        count += 1
        if limit and count >= limit:
            break


c:\Homework\Code File\Python Code File\Pratice\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\minht\.cache\huggingface\hub\datasets--HuggingFaceFW--fineweb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


### Train tokenizer

In [3]:
tokenizer = Tokenizer(models.BPE(byte_fallback=True))
tokenizer.normalizer = normalizers.NFKC()
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True, use_regex=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=25_000,
    min_frequency=2,
    special_tokens=["[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    show_progress=True,
)

texts = get_texts(dataset, limit=10_000)
tokenizer.train_from_iterator(texts, trainer=trainer)
tokenizer.save("bpe_tokenizer.json")

### Load tokenizer and test

In [3]:
tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

text = "Let's go play some games?"
enc = tokenizer.encode(text)
print("Token IDs: ", enc.ids)
print("Decoded: ", tokenizer.decode(enc.ids))

Token IDs:  [3548, 277, 396, 1260, 1453, 640, 9702, 34]
Decoded:   Let's go play some games?
